# Feature-set similarity effect analysis

This read-only analysis compares `scalar_compact` with other feature sets under matched evaluation conditions. It visualizes overall gains and whether gains are larger for smaller scorer LLMs.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from notebooks.utils.classification_results_analysis_utils import load_classification_run
from notebooks.utils.feature_set_similarity_effect_utils import (
    DEFAULT_MODEL_SIZES_BILLIONS, aggregate_feature_set_deltas, append_external_feature_set_comparison, attach_model_sizes,
    gpt_who_experiment_for_training_regime,
    matched_feature_set_deltas, plot_delta_by_model_size, plot_feature_set_comparison,
    plot_feature_set_delta_heatmap,
)
from notebooks.utils.gpt_who_results_analysis_utils import load_gpt_who_metrics

DATA_ROOT = Path(os.environ.get('TEXT_DETECTION_DATA', 'D:/Projects/master_diploma/text-detection-data'))
DEFAULT_SOURCE_RUN_ID = 'feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100'
CLASSIFIER_ANALYSIS_SOURCE_RUN_ID = os.environ.get('CLASSIFIER_ANALYSIS_SOURCE_RUN_ID', DEFAULT_SOURCE_RUN_ID)
CLASSIFIER_ANALYSIS_RUN_IDS = os.environ.get('CLASSIFIER_ANALYSIS_RUN_IDS', CLASSIFIER_ANALYSIS_SOURCE_RUN_ID)
RUN_IDS = [run_id.strip() for run_id in CLASSIFIER_ANALYSIS_RUN_IDS.split(',') if run_id.strip()]
BASELINE_FEATURE_SET = os.environ.get('FEATURE_EFFECT_BASELINE', 'scalar_compact')
COMPARISON_FEATURE_SETS = tuple(name.strip() for name in os.environ.get('FEATURE_EFFECT_COMPARISONS', 'simple_similarity,extended,scalar_all').split(',') if name.strip())
METRIC = os.environ.get('FEATURE_EFFECT_METRIC', 'macro_f1')
OUTPUT_ROOT = Path(os.environ.get('FEATURE_EFFECT_OUTPUT_ROOT', str(DATA_ROOT / 'experiment-results' / 'classification' / 'analysis' / 'feature-set-similarity-effect')))
GPT_WHO_RESULTS_ROOT = DATA_ROOT / 'experiment-results' / 'gpt-who-baseline'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
frames = []
warnings = []
for run_id in RUN_IDS:
    try:
        run = load_classification_run(DATA_ROOT, run_id=run_id)
    except (OSError, ValueError) as error:
        warnings.append(f'{run_id}: {error}')
        continue
    metrics = run.tables.get('evaluation_metrics', pd.DataFrame()).copy()
    if not metrics.empty:
        metrics['source_run_id'] = run_id
        frames.append(metrics)
metrics = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if warnings:
    display(pd.DataFrame({'warning': warnings}))
if metrics.empty:
    raise ValueError('No evaluation_metrics rows were loaded from the configured run IDs.')
available_feature_sets = sorted(metrics['feature_set'].dropna().astype(str).unique())
comparison_feature_sets = tuple(name for name in COMPARISON_FEATURE_SETS if name in available_feature_sets)
missing_feature_sets = sorted(set(COMPARISON_FEATURE_SETS).difference(comparison_feature_sets))
if missing_feature_sets:
    display(pd.DataFrame({'missing_comparison_feature_set': missing_feature_sets}))
gpt_who_metrics = load_gpt_who_metrics(GPT_WHO_RESULTS_ROOT)
training_regimes = sorted(metrics['training_regime'].dropna().astype(str).unique())
print(f'Loaded {len(metrics):,} evaluation rows for {metrics["scorer_llm"].nunique()} scorer LLMs.')


In [ ]:
context_results = {}
for training_regime in training_regimes:
    context_metrics = metrics.loc[metrics['training_regime'].eq(training_regime)].copy()
    deltas = matched_feature_set_deltas(
        context_metrics,
        baseline_feature_set=BASELINE_FEATURE_SET,
        comparison_feature_sets=comparison_feature_sets,
        metric=METRIC,
    )
    gpt_who_experiment = gpt_who_experiment_for_training_regime(training_regime)
    context_gpt_who = gpt_who_metrics.loc[
        gpt_who_metrics['experiment_name'].eq(gpt_who_experiment)
    ] if gpt_who_experiment and not gpt_who_metrics.empty else pd.DataFrame()
    if not context_gpt_who.empty:
        deltas = pd.concat([
            deltas,
            append_external_feature_set_comparison(
                context_metrics, context_gpt_who,
                baseline_feature_set=BASELINE_FEATURE_SET,
                external_feature_set='gpt_who', metric=METRIC,
            ),
        ], ignore_index=True)
    summary = attach_model_sizes(aggregate_feature_set_deltas(deltas))
    coverage = (
        context_metrics.groupby(['scorer_llm', 'feature_set'], as_index=False)
        .size().rename(columns={'size': 'evaluation_row_count'})
    )
    training_regime_output_root = OUTPUT_ROOT / training_regime.replace('+', '_plus_')
    training_regime_output_root.mkdir(parents=True, exist_ok=True)
    deltas.to_csv(training_regime_output_root / 'matched_feature_set_deltas.csv', index=False)
    summary.to_csv(training_regime_output_root / 'feature_set_effect_summary.csv', index=False)
    coverage.to_csv(training_regime_output_root / 'feature_set_coverage.csv', index=False)
    context_results[training_regime] = {
        'deltas': deltas, 'summary': summary,
        'output_root': training_regime_output_root,
        'gpt_who_experiment': gpt_who_experiment,
    }
    display(pd.DataFrame([{
        'training_regime': training_regime,
        'gpt_who_experiment': gpt_who_experiment or 'not available',
        'evaluation_rows': len(context_metrics),
    }]))
    display(summary)


In [ ]:
for training_regime, result in context_results.items():
    deltas = result['deltas']
    summary = result['summary']
    figures = {
        'feature_set_comparison.png': plot_feature_set_comparison(
            deltas, baseline_feature_set=BASELINE_FEATURE_SET,
            title=f'{METRIC} comparison - {training_regime}',
        ),
        'feature_set_delta_heatmap.png': plot_feature_set_delta_heatmap(
            summary, title=f'{METRIC} gain over {BASELINE_FEATURE_SET} - {training_regime}',
        ),
        'delta_by_model_size.png': plot_delta_by_model_size(
            summary, title=f'{METRIC} gain versus scorer model size - {training_regime}',
        ),
    }
    for filename, figure in figures.items():
        display(figure)
        figure.savefig(result['output_root'] / filename, dpi=160, bbox_inches='tight')
        plt.close(figure)
    unknown_sizes = sorted(set(summary.loc[~summary['size_known'], 'scorer_llm'])) if not summary.empty else []
    if unknown_sizes:
        display(pd.DataFrame({'unknown_model_size': unknown_sizes, 'training_regime': training_regime}))
    print(f'Wrote {training_regime} analysis outputs to {result["output_root"]}')
